In [1]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

In [6]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),  
    password=os.getenv("NEO4J_PASSWORD"),
        database="neo4j-inflearn"

) 

In [7]:
# 테스트 쿼리 실행 
cypher_query = """
MATCH (n:Movie)
RETURN COUNT(n) AS Movie_Count
"""

graph.query(cypher_query)

[{'Movie_Count': 4803}]

In [8]:
fulltext_index_query="""  
create fulltext index movie_title_fulltext if not exists
for (m:Movie) on each [m.title]
"""
graph.query(fulltext_index_query)

[]

In [9]:
graph.query("show fulltext indexes")

[{'id': 8,
  'name': 'movie_title_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['title'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [11]:
fulltext_index_query="""  
create fulltext index movie_title_tagline_fulltext if not exists
for (m:Movie) on each [m.title,m.tagline]
"""
graph.query(fulltext_index_query)

[]

In [12]:
graph.query("show fulltext indexes")

[{'id': 8,
  'name': 'movie_title_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['title'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 9,
  'name': 'movie_title_tagline_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['title', 'tagline'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [14]:
person_fulltext_index_query="""   
create fulltext index person_name_fulltext if not exists
for (p:Person) on each [p.name]
"""
graph.query(person_fulltext_index_query)

[]

In [18]:
# `love`라는 단어가 포함된 영화 제목을 검색
# 전문 검색 인덱스를 사용하여 영화 제목에서 'love' 키워드 검색
search_query = """
// db.index.fulltext.queryNodes: 전문 검색 인덱스를 사용하여 노드를 검색하는 프로시저
// "movie_title_fulltext": 앞서 생성한 영화 제목 전문 검색 인덱스 이름
// $search_term: 파라미터로 전달되는 검색어 (여기서는 "love")
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)

// YIELD: 프로시저의 결과를 반환
// node: 검색된 노드 객체, score: 검색 관련성 점수
YIELD node, score

// RETURN: 결과로 반환할 데이터 지정
// node.title: 검색된 영화 노드의 제목 속성
// AS MovieTitle: 결과 컬럼명 지정
// score AS SearchRelevance: 검색 관련성 점수를 SearchRelevance로 이름 지정
RETURN node.title AS MovieTitle, score AS SearchRelevance

// ORDER BY: 결과 정렬 방식 지정
// SearchRelevance DESC: 관련성 점수 기준 내림차순 정렬(가장 관련성 높은 결과가 먼저 표시)
ORDER BY SearchRelevance DESC

// LIMIT 5: 상위 5개 결과만 반환
LIMIT 5
"""
graph.query("show fulltext index")

[{'id': 8,
  'name': 'movie_title_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['title'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 9,
  'name': 'movie_title_tagline_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['title', 'tagline'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 10,
  'name': 'person_name_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['Person'],
  'properties': ['name'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [20]:
# `love`라는 단어가 포함된 영화 제목을 검색
# 전문 검색 인덱스를 사용하여 영화 제목에서 'love' 키워드 검색
search_query = """
// db.index.fulltext.queryNodes: 전문 검색 인덱스를 사용하여 노드를 검색하는 프로시저
// "movie_title_fulltext": 앞서 생성한 영화 제목 전문 검색 인덱스 이름
// $search_term: 파라미터로 전달되는 검색어 (여기서는 "love")
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)

// YIELD: 프로시저의 결과를 반환
// node: 검색된 노드 객체, score: 검색 관련성 점수
YIELD node, score

// RETURN: 결과로 반환할 데이터 지정
// node.title: 검색된 영화 노드의 제목 속성
// AS MovieTitle: 결과 컬럼명 지정
// score AS SearchRelevance: 검색 관련성 점수를 SearchRelevance로 이름 지정
RETURN node.title AS MovieTitle, score AS SearchRelevance

// ORDER BY: 결과 정렬 방식 지정
// SearchRelevance DESC: 관련성 점수 기준 내림차순 정렬(가장 관련성 높은 결과가 먼저 표시)
ORDER BY SearchRelevance DESC

// LIMIT 5: 상위 5개 결과만 반환
LIMIT 5
"""
graph.query(search_query, params={"search_term":"love"})

[{'MovieTitle': 'Love Stinks', 'SearchRelevance': 2.281766891479492},
 {'MovieTitle': 'Love & Basketball', 'SearchRelevance': 2.281766891479492},
 {'MovieTitle': 'Love Happens', 'SearchRelevance': 2.281766891479492},
 {'MovieTitle': 'Love Ranch', 'SearchRelevance': 2.281766891479492},
 {'MovieTitle': 'Love Letters', 'SearchRelevance': 2.281766891479492}]

In [21]:
search_query="""  
CALL db.index.fulltext.queryNodes("person_name_fulltext",$search_term)
YIELD node,score

RETURN node.name AS PersonName, score AS SearchRElevance
ORDER BY SearchRElevance DESC
LIMIT 3
"""
graph.query(search_query,params={"search_term":"tom"})

[{'PersonName': 'Tom Noonan', 'SearchRElevance': 2.360572099685669},
 {'PersonName': 'Tom Hulce', 'SearchRElevance': 2.360572099685669},
 {'PersonName': 'Tom Bosley', 'SearchRElevance': 2.360572099685669}]

In [22]:
# 영화 제목과 태그라인에서 키워드 검색
search_query = """
// db.index.fulltext.queryNodes: 전문 검색 인덱스를 사용하여 노드를 검색하는 프로시저
// "movie_title_tagline_fulltext": 영화 제목과 태그라인을 위한 전문 검색 인덱스 이름
CALL db.index.fulltext.queryNodes("movie_title_tagline_fulltext", $search_term)

// YIELD: 프로시저의 결과를 반환
// node: 검색된 노드 객체, score: 검색 관련성 점수
YIELD node, score

// RETURN: 결과로 반환할 데이터 지정
// node.title: 검색된 영화의 제목
// node.tagline: 검색된 영화의 태그라인
// score: 검색 관련성 점수
RETURN node.title AS MovieTitle, node.tagline AS Tagline, score AS SearchRelevance

// ORDER BY: 결과 정렬 방식 지정
// SearchRelevance DESC: 관련성 점수 기준 내림차순 정렬
ORDER BY SearchRelevance DESC

// LIMIT 5: 상위 5개 결과만 반환
LIMIT 5
"""
graph.query(search_query, params={"search_term": "dream"})


[{'MovieTitle': 'Goal!: The Dream Begins',
  'Tagline': 'Every Dream Has A Beginning',
  'SearchRelevance': 4.898340702056885},
 {'MovieTitle': 'Opal Dream',
  'Tagline': None,
  'SearchRelevance': 3.1974334716796875},
 {'MovieTitle': 'Dream House',
  'Tagline': 'Once upon a time, there were two little girls who lived in a house.',
  'SearchRelevance': 3.1974334716796875},
 {'MovieTitle': 'The Walk',
  'Tagline': 'Dream High.',
  'SearchRelevance': 3.0625619888305664},
 {'MovieTitle': 'Superstar',
  'Tagline': 'Dare to dream.',
  'SearchRelevance': 2.84822940826416}]

In [23]:
# 퍼지 검색 - 오타를 허용하는 검색 (예: 'afair'로 'affair' 찾기)
fuzzy_search_query = """
// db.index.fulltext.queryNodes: 전문 검색 인덱스를 사용하여 노드를 검색하는 프로시저
// "movie_title_fulltext": 영화 제목에 대한 전문 검색 인덱스 이름
// $search_term: 파라미터로 전달된 검색어 (여기서는 "afair~0.7"로 전달됨)
// ~0.7: 퍼지 검색 표시자로, 검색어와 70% 이상 일치하는 결과를 반환
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)

// YIELD: 프로시저의 결과를 반환
// node: 검색된 노드 객체, score: 검색 관련성 점수
YIELD node, score

// RETURN: 결과로 반환할 데이터 지정
// node.title: 검색된 영화 노드의 제목 속성
// AS MovieTitle: 결과 컬럼명을 MovieTitle로 지정
// score AS SearchRelevance: 검색 관련성 점수를 SearchRelevance로 이름 지정
RETURN node.title AS MovieTitle, score AS SearchRelevance

// ORDER BY: 결과 정렬 방식 지정
// SearchRelevance DESC: 관련성 점수 기준 내림차순 정렬(가장 관련성 높은 결과가 먼저 표시)
ORDER BY SearchRelevance DESC

// LIMIT 5: 상위 5개 결과만 반환
LIMIT 5
"""

# graph.query: Neo4j 데이터베이스에 쿼리를 실행하는 함수
# params={"search_term": "afair~0.7"}: 퍼지 검색을 위한 파라미터 전달
# ~0.7은 편집 거리(edit distance)를 기반으로 70% 유사도를 가진 결과까지 포함
fuzzy_results = graph.query(fuzzy_search_query, params={"search_term": "afair~0.7"})

# 검색 결과를 순회하며 영화 제목과 관련도 점수를 출력
for result in fuzzy_results:
    print(f"{result['MovieTitle']} (관련도: {result['SearchRelevance']})")

Vanity Fair (관련도: 2.685884475708008)
Fair Game (관련도: 2.685884475708008)
State Fair (관련도: 2.685884475708008)
The Thomas Crown Affair (관련도: 2.3315298557281494)
My Fair Lady (관련도: 2.3031468391418457)


In [24]:
# 와일드카드 검색 - 특정 패턴으로 시작하는 단어 검색 (예: 'star'로 시작하는 영화)
wildcard_search_query = """
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)
YIELD node, score

RETURN node.title AS MovieTitle, node.released AS ReleaseDate, score AS SearchRelevance
ORDER BY SearchRelevance DESC
LIMIT 5
"""

# star*: 'star'로 시작하는 모든 단어를 검색
wildcard_results = graph.query(wildcard_search_query, params={"search_term": "star*"})

print("\n=== 와일드카드 검색 결과 (star*) ===")
for result in wildcard_results:
    print(f"{result['MovieTitle']} ({result['ReleaseDate']}) - 관련도: {result['SearchRelevance']:.4f}")


=== 와일드카드 검색 결과 (star*) ===
Star Trek Beyond (2016-07-07) - 관련도: 1.0000
The Fault in Our Stars (2014-05-16) - 관련도: 1.0000
My Lucky Star (2013-09-17) - 관련도: 1.0000
20 Feet from Stardom (2013-06-14) - 관련도: 1.0000
Star Wars (1977-05-25) - 관련도: 1.0000


In [25]:
# 정확한 구문 검색 - 정확한 문구를 검색 (예: "love story")
phrase_search_query = """
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)
YIELD node, score

RETURN node.title AS MovieTitle, node.released AS ReleaseDate, score AS SearchRelevance
ORDER BY SearchRelevance DESC
LIMIT 5
"""

# \"love story\": 정확히 "love story"라는 구문을 포함하는 영화 검색
# 이스케이프 처리된 쌍따옴표를 사용하여 정확한 구문을 지정
phrase_results = graph.query(phrase_search_query, params={"search_term": "\"love story\""})

print("\n=== 정확한 구문 검색 결과 (\"love story\") ===")
for result in phrase_results:
    print(f"{result['MovieTitle']} ({result['ReleaseDate']}) - 관련도: {result['SearchRelevance']:.4f}")


=== 정확한 구문 검색 결과 ("love story") ===
Wristcutters: A Love Story (2006-01-24) - 관련도: 3.6628
Capitalism: A Love Story (2009-09-06) - 관련도: 3.6628


In [26]:
# 논리 연산자 검색 - 특정 단어를 포함하고 다른 단어는 제외 (예: 'love'와 'story'를 포함하고 'horror'는 제외)
boolean_search_query = """
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)
YIELD node, score

RETURN node.title AS MovieTitle, node.released AS ReleaseDate, score AS SearchRelevance
ORDER BY SearchRelevance DESC
LIMIT 5
"""

# love AND story NOT horror: 'love'와 'story'를 포함하고 'horror'는 제외하는 검색
boolean_results = graph.query(boolean_search_query, params={"search_term": "love AND story NOT horror"})

print("\n=== 논리 연산자 검색 결과 (love AND story NOT horror) ===")
for result in boolean_results:
    print(f"{result['MovieTitle']} ({result['ReleaseDate']}) - 관련도: {result['SearchRelevance']:.4f}")


=== 논리 연산자 검색 결과 (love AND story NOT horror) ===
Wristcutters: A Love Story (2006-01-24) - 관련도: 3.6628
Capitalism: A Love Story (2009-09-06) - 관련도: 3.6628


In [27]:
# 가중치 부여 검색 - 특정 단어에 더 높은 가중치 부여 (예: 'love'에 4배 가중치)
boost_search_query = """
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term)
YIELD node, score

RETURN node.title AS MovieTitle, node.released AS ReleaseDate, score AS SearchRelevance
ORDER BY SearchRelevance DESC
LIMIT 5
"""

# love^4 story: 'love'에 4배 가중치를 부여한 검색
boost_results = graph.query(boost_search_query, params={"search_term": "love^4 story"})

print("\n=== 가중치 부여 검색 결과 (love^4 story) ===")
for result in boost_results:
    print(f"{result['MovieTitle']} ({result['ReleaseDate']}) - 관련도: {result['SearchRelevance']:.4f}")


=== 가중치 부여 검색 결과 (love^4 story) ===
Love Jones (1997-03-14) - 관련도: 9.1271
Love Actually (2003-09-07) - 관련도: 9.1271
Love Ranch (2010-06-30) - 관련도: 9.1271
Endless Love (2014-02-12) - 관련도: 9.1271
Love Letters (1983-04-01) - 관련도: 9.1271


In [28]:
# 전문 검색 -> 그래프 탐색: 특정 단어를 포함하는 영화를 먼저 찾고(full-text), 각 영화에 출연한 출연 배우 찾기(graph)
combined_search_query = """
// db.index.fulltext.queryNodes: 전문 검색 인덱스를 사용하여 노드 검색
// "movie_title_fulltext": 영화 제목에 대한 전문 검색 인덱스 이름
// $search_term: 파라미터로 전달된 검색어
CALL db.index.fulltext.queryNodes("movie_title_fulltext", $search_term) 

// YIELD: 프로시저의 결과를 반환
// node as movie: 검색된 노드를 movie로 별칭 지정
// score: 검색 관련성 점수
YIELD node as movie, score

// MATCH: 그래프 패턴 매칭
// (movie)<-[:ACTED_IN]-(actor:Person): movie 노드에 ACTED_IN 관계로 연결된 Person 타입의 actor 노드 찾기
MATCH (movie)<-[:ACTED_IN]-(actor:Person)

// RETURN: 쿼리 결과 반환
// movie.title AS MovieTitle: 영화 제목을 MovieTitle로 별칭 지정
// score AS SearchRelevance: 검색 관련성 점수를 SearchRelevance로 별칭 지정
// collect(actor.name) AS Actors: 각 영화에 출연한 모든 배우 이름을 배열로 수집하여 Actors로 별칭 지정
RETURN movie.title AS MovieTitle, score AS SearchRelevance, 
       collect(actor.name) AS Actors

// ORDER BY SearchRelevance DESC: 검색 관련성 점수 기준 내림차순 정렬(가장 관련성 높은 결과가 먼저 표시)
ORDER BY SearchRelevance DESC

// LIMIT 5: 상위 5개 결과만 반환
LIMIT 5
"""

# graph.query: Neo4j 데이터베이스에 쿼리를 실행하는 함수
# params={"search_term": "love"}: "love"라는 검색어를 파라미터로 전달
combined_results = graph.query(combined_search_query, params={"search_term": "love"})

# 검색 결과를 순회하며 영화 제목, 관련도 점수, 출연 배우 목록을 출력
for result in combined_results:
    print(f"{result['MovieTitle']} (관련도: {result['SearchRelevance']})")
    print(f"  출연 배우: {', '.join(result['Actors'])}")
    print()

Love Stinks (관련도: 2.281766891479492)
  출연 배우: Bridgette Wilson, Bill Bellamy, Tiffani Thiessen, French Stewart, Tyra Banks

Love & Basketball (관련도: 2.281766891479492)
  출연 배우: Alfre Woodard, Omar Epps, Sanaa Lathan, Chris Warren, Jr., Kyla Pratt

Love Happens (관련도: 2.281766891479492)
  출연 배우: Martin Sheen, Jennifer Aniston, Aaron Eckhart, Judy Greer, Deirdre Blades

Love Ranch (관련도: 2.281766891479492)
  출연 배우: Joe Pesci, Bai Ling, Gina Gershon, Helen Mirren, Sergio Peris-Mencheta

Love Letters (관련도: 2.281766891479492)
  출연 배우: Jamie Lee Curtis, James Keach, Matt Clark, Bonnie Bartlett, Amy Madigan

